# Certificat de navigation — Validation empirique sur GPT-2

**Question testée :** les prompts qui mènent à des hallucinations s'attardent-ils
réellement plus longtemps dans une zone ambiguë (« col ») entre les couches d'un
vrai Transformer, comparé aux prompts factuels ?

**Historique des corrections intégrées dans ce notebook** (à ne pas refaire) :
1. Bug de forme (`factual_centroid` vs `trajectory_pca.flatten()`) — corrigé : comparaison symétrique moyennée sur les couches.
2. PCA et centroïdes recalculés **par pli de validation croisée**, jamais sur l'ensemble train+test (fuite de données).
3. Formule du `saddle_score` — la version *« vitesse modérée »* (`1 - |vel_norm-0.5|*2`) a été testée avec un contrôle de recouvrement contre le vrai palier injecté : **0 couche en commun sur 9/10 échantillons synthétiques**. Elle « discriminait » les classes en captant un artefact du générateur, pas le phénomène recherché. Remplacée par *« vitesse faible »* (`1 - vel_norm`), qui recoupe réellement le palier injecté.
4. Le dwell time est maintenant testé **seul, isolé des 9 autres features** — un Random Forest complet peut afficher AUC=1.0 sans jamais utiliser le dwell si d'autres features suffisent ; ça masquerait un dwell cassé. Ce notebook rapporte les deux.

**Étape 0 ci-dessous est un garde-fou obligatoire** : si le contrôle positif/négatif/recouvrement échoue sur données synthétiques, le notebook s'arrête avant de toucher GPT-2 — pas la peine de dépenser du GPU sur une feature qui ne mesure pas ce qu'elle prétend mesurer.


---

### ⚠️ Mise à jour v2 — normalisation du dwell time

Le premier run réel sur GPT-2 a montré `max_dwell_time` **constant** (std = 0.000)
sur les 20 prompts, factuels et hallucinatoires confondus — la feature ne
discriminait rien du tout (AUC dwell seul = 0.500, importance = 0.000), alors
que le vecteur complet obtenait AUC = 0.890 (p = 0.005) grâce à `mean_curvature`
et aux distances aux centroïdes.

**Diagnostic** : la normalisation min-max était calculée *par trajectoire*
individuelle. Si les toutes premières couches de GPT-2 subissent une
restructuration forte et **universelle** (indépendante du contenu — phénomène
documenté dans les Transformers), cette volatilité précoce écrase, après
normalisation par trajectoire, toute variation liée au contenu dans les
couches suivantes — et sature le dwell à la même valeur pour tout le monde.

**Correctifs appliqués ci-dessous** :
1. Fenêtre restreinte aux couches 4-12 (on ignore la restructuration précoce connue)
2. Normalisation **globale, fold-safe** (calculée sur le train set du pli, pas par trajectoire individuelle) — évite le problème de fond d'une normalisation auto-référentielle qui écrase par construction les différences inter-échantillons

**Réserve honnête** : une reproduction synthétique de ce problème n'a que
partiellement répliqué l'échec observé sur GPT-2 (la version originale gardait
un peu de variance sur mes données synthétiques, alors qu'elle était
totalement dégénérée sur les vraies données — le vrai phénomène GPT-2 est donc
probablement plus systématique que ce qu'un générateur synthétique articifiel
reproduit facilement). Le correctif s'attaque au bon mécanisme mais n'est
confirmé qu'après un nouveau run réel — regarde `max_dwell_time std` dans les
résultats : s'il est encore ≈0, le problème n'est pas résolu et il faudra
regarder plus loin (ex: la restructuration s'étend au-delà de la couche 4-5).


---

### ⚠️ Mise à jour v3 — dwell relatif à la population (pas à soi-même)

Le diagnostic brut (couche par couche, sur 4 vraies trajectoires GPT-2) a
confirmé la cause exacte de la saturation : GPT-2 a une **forme universelle**
de courbure/vitesse par couche — haute aux couches 0-1, un creux net aux
couches 3-4, une remontée progressive, une explosion aux couches 10-12 —
**identique pour les prompts factuels ET hallucinatoires**. Ce n'est pas un
artefact de calcul, c'est visible directement dans les nombres bruts. Le
"col" tel que défini (faible courbure+vitesse) correspond exactement à ce
creux universel des couches 3-4, présent chez tout le monde -> ne peut rien
discriminer par construction, peu importe la fenêtre de couches choisie.

**Correctif v3** : ne plus mesurer la courbure/vitesse en valeur absolue,
mais leur **écart par rapport à la moyenne de la population** (calculée sur
le train set de chaque pli, fold-safe) à CHAQUE couche. Une couche est "au
col" si elle est significativement (>1 écart-type) EN DESSOUS du gabarit
commun à cette couche précise -- pas si elle est basse dans l'absolu.

**Validation synthétique v3** : cette fois le générateur synthétique reproduit
le vrai échec (gabarit calqué sur les valeurs GPT-2 réellement observées,
méthode absolue -> std quasi nul, AUC=0.500, comme sur les vraies données) et
le correctif le corrige (std=1.755, AUC=1.000, recouvrement=100%, contrôle
négatif=0.624). C'est la première fois que la reproduction synthétique
réplique fidèlement l'échec réel avant de proposer un fix -- confiance plus
solide que les deux tentatives précédentes.


In [ ]:
# Installation (Colab : redémarrer le runtime si demandé après l'install)
!pip install -q torch transformers scikit-learn numpy scipy matplotlib


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score
import json, os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cpu':
    print("⚠️  Pas de GPU détecté. Menu Colab -> Exécution -> Modifier le type d'exécution -> GPU, pour aller plus vite.")


## Étape 0 — Garde-fou : validation synthétique du saddle_score (obligatoire)

On génère des trajectoires synthétiques (forme identique à GPT-2 : 13 couches) où
on **connaît exactement** où se trouve le vrai palier de dwell injecté. Trois
vérifications, pas seulement une accuracy globale :

1. **Contrôle positif** : le classifieur détecte-t-il un signal quand il y en a un ?
2. **Contrôle négatif** (labels permutés) : retombe-t-il au hasard quand il n'y a rien à trouver ?
3. **Recouvrement** : les couches que `saddle_score` flague comme « au col » correspondent-elles
   vraiment au palier injecté, ou à un artefact du générateur qui coïncide par hasard avec le label ?
   (C'est ce 3e test qui a fait échouer la version précédente de cette feature.)

Si l'un des trois échoue, le notebook s'arrête ici — voir le message d'erreur pour la marche à suivre.


In [ ]:
def population_curv_vel_stats(train_trajs_pca):
    """Moyenne/écart-type de courbure et vitesse, COUCHE PAR COUCHE, sur le
    train set du pli (fold-safe) -- c'est le gabarit commun contre lequel on
    va comparer chaque trajectoire individuelle."""
    all_curv, all_vel = [], []
    for t in train_trajs_pca:
        fd = np.gradient(t, axis=0)
        sd = np.gradient(fd, axis=0)
        all_curv.append(np.linalg.norm(sd, axis=1))
        all_vel.append(np.linalg.norm(np.gradient(t, axis=0), axis=1))
    all_curv = np.array(all_curv)  # (n_train, n_layers)
    all_vel = np.array(all_vel)
    return all_curv.mean(axis=0), all_curv.std(axis=0) + 1e-8, all_vel.mean(axis=0), all_vel.std(axis=0) + 1e-8


def saddle_score_dwell(trajectory_pca, pop_curv_mean, pop_curv_std, pop_vel_mean, pop_vel_std, z_threshold=1.0):
    """Dwell time au col -- v3 : une couche est "au col" si sa courbure ET sa
    vitesse sont significativement EN DESSOUS du gabarit de la population à
    cette couche précise (z-score < -z_threshold), pas juste basses dans
    l'absolu. Ça retire l'effet du gabarit universel (creux commun couches
    3-4) et ne garde que ce qui s'écarte réellement, couche par couche."""
    first_deriv = np.gradient(trajectory_pca, axis=0)
    second_deriv = np.gradient(first_deriv, axis=0)
    curvature = np.linalg.norm(second_deriv, axis=1)
    velocity = np.linalg.norm(np.gradient(trajectory_pca, axis=0), axis=1)

    cz = (curvature - pop_curv_mean) / pop_curv_std
    vz = (velocity - pop_vel_mean) / pop_vel_std
    anomaly = (-cz) + (-vz)  # élevé si nettement EN DESSOUS du gabarit commun

    above = anomaly > z_threshold
    dwell_times, cur = [], 0
    for v in above:
        cur = cur + 1 if v else 0
        if cur > 0:
            dwell_times.append(cur)
    max_dwell = max(dwell_times) if dwell_times else 0
    mean_dwell = np.mean(dwell_times) if dwell_times else 0
    flagged_layers = set(np.where(above)[0].tolist())
    return max_dwell, mean_dwell, flagged_layers


UNIV_CURV = np.array([26, 25, 11, 2, 1.5, 4, 7, 10, 12, 30, 80, 170, 185])
UNIV_VEL = np.array([52, 26, 4.5, 5.5, 6, 7, 11, 20, 28, 40, 80, 145, 300])


def generate_synthetic_trajectory(regime="factual", dwell_at_saddle=0, rng=None, class_signal_strength=0.3):
    """Gabarit calqué sur les vraies valeurs de courbure/vitesse GPT-2
    observées au diagnostic (couche 0-12), + signal de classe optionnel :
    une réduction modérée (pas une chute à zéro) de courbure/vitesse sur un
    palier, pour simuler un dwell réaliste, pas artificiellement facile."""
    rng = rng or np.random
    curv = UNIV_CURV * (1 + rng.randn(N_LAYERS) * 0.08)
    vel = UNIV_VEL * (1 + rng.randn(N_LAYERS) * 0.08)
    true_dwell_range = None
    if regime == "hallucination" and dwell_at_saddle > 0:
        start = rng.randint(3, 7)
        true_dwell_range = (start, min(start + dwell_at_saddle, N_LAYERS))
        for l in range(*true_dwell_range):
            curv[l] *= class_signal_strength
            vel[l] *= class_signal_strength
    return curv, vel, true_dwell_range


N_LAYERS = 13


def run_synthetic_gate():
    rng = np.random.RandomState(RANDOM_STATE)
    curvs, vels, labels, true_ranges = [], [], [], []
    for _ in range(30):
        c, v, _ = generate_synthetic_trajectory("factual", 0, rng=rng)
        curvs.append(c); vels.append(v); labels.append(0); true_ranges.append(None)
    for _ in range(30):
        d = rng.randint(3, 6)
        c, v, r = generate_synthetic_trajectory("hallucination", d, rng=rng)
        curvs.append(c); vels.append(v); labels.append(1); true_ranges.append(r)
    labels = np.array(labels)

    def dwell_from_curv_vel(curv, vel, pcm, pcs, pvm, pvs, z_threshold=1.0):
        cz = (curv - pcm) / pcs
        vz = (vel - pvm) / pvs
        anomaly = (-cz) + (-vz)
        above = anomaly > z_threshold
        dwell, cur = [], 0
        for v_ in above:
            cur = cur + 1 if v_ else 0
            if cur > 0:
                dwell.append(cur)
        return (max(dwell) if dwell else 0), set(np.where(above)[0].tolist())

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    max_dwells, flagged_all = [None] * 60, [None] * 60
    for train_idx, test_idx in skf.split(np.zeros(60), labels):
        pop_curv = np.array([curvs[i] for i in train_idx])
        pop_vel = np.array([vels[i] for i in train_idx])
        pcm, pcs = pop_curv.mean(axis=0), pop_curv.std(axis=0) + 1e-8
        pvm, pvs = pop_vel.mean(axis=0), pop_vel.std(axis=0) + 1e-8
        for i in test_idx:
            md, flagged = dwell_from_curv_vel(curvs[i], vels[i], pcm, pcs, pvm, pvs)
            max_dwells[i] = md
            flagged_all[i] = flagged

    X = np.array(max_dwells).reshape(-1, 1)
    skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    probas = np.zeros(60)
    for tr, te in skf2.split(X, labels):
        clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE).fit(X[tr], labels[tr])
        probas[te] = clf.predict_proba(X[te])[:, 1]
    auc_pos = roc_auc_score(labels, probas)

    labels_shuf = rng.permutation(labels)
    probas_neg = np.zeros(60)
    for tr, te in skf2.split(X, labels_shuf):
        clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE).fit(X[tr], labels_shuf[tr])
        probas_neg[te] = clf.predict_proba(X[te])[:, 1]
    auc_neg = roc_auc_score(labels_shuf, probas_neg)

    overlaps = []
    for i in range(60):
        if true_ranges[i] is None:
            continue
        true_layers = set(range(*true_ranges[i]))
        overlaps.append(len(flagged_all[i] & true_layers) > 0)
    overlap_rate = np.mean(overlaps)
    std_check = X.std()

    print(f"Contrôle positif (dwell seul)       : AUC = {auc_pos:.3f}   {'✅' if auc_pos > 0.75 else '❌'}")
    print(f"Contrôle négatif (labels permutés)  : AUC = {auc_neg:.3f}   {'✅' if auc_neg < 0.75 else '❌'}")
    print(f"Recouvrement avec le vrai palier    : {overlap_rate:.0%} des échantillons  {'✅' if overlap_rate > 0.5 else '❌'}")
    print(f"Écart-type de max_dwell             : {std_check:.3f}   {'✅ (non degenere)' if std_check > 0.3 else '⚠️ proche de 0'}")

    passed = auc_pos > 0.75 and auc_neg < 0.75 and overlap_rate > 0.5 and std_check > 0.3
    if not passed:
        raise RuntimeError(
            "GARDE-FOU ECHOUE -- le dwell relatif a la population ne mesure pas de "
            "façon fiable le phénomène recherché. Ne pas continuer sur GPT-2."
        )
    print("\n🟢 Garde-fou passé (v3, dwell relatif à la population) -- on continue sur GPT-2.")
    return True

run_synthetic_gate()


## Étape 1 — Dataset et extraction des hidden states GPT-2

50 prompts annotés (25 factuels vérifiables, 25 hallucinatoires connus) --
élargi depuis la version à 20 prompts pour donner assez de puissance
statistique au sous-ensemble de features à signal réel identifié à l'étape
3bis (`mean_curvature`, `dist_factual`, `max_velocity`, `ratio_dist`).


In [ ]:
PROMPTS = [
    # === FACTUELS (label=0) -- 25 ===
    {"text": "The capital of France is", "label": 0},
    {"text": "Water boils at 100 degrees Celsius at sea level.", "label": 0},
    {"text": "The speed of light in vacuum is approximately 299,792 kilometers per second.", "label": 0},
    {"text": "The Earth orbits around the Sun.", "label": 0},
    {"text": "Shakespeare wrote Hamlet.", "label": 0},
    {"text": "The chemical formula for water is H2O.", "label": 0},
    {"text": "The Great Wall of China is located in China.", "label": 0},
    {"text": "Pi is approximately 3.14159.", "label": 0},
    {"text": "The human heart has four chambers.", "label": 0},
    {"text": "World War II ended in 1945.", "label": 0},
    {"text": "The largest planet in the solar system is Jupiter.", "label": 0},
    {"text": "Mount Everest is the tallest mountain above sea level.", "label": 0},
    {"text": "The currency used in Japan is the yen.", "label": 0},
    {"text": "Photosynthesis converts sunlight into chemical energy in plants.", "label": 0},
    {"text": "The human body has 206 bones in adulthood.", "label": 0},
    {"text": "The Pacific Ocean is the largest ocean on Earth.", "label": 0},
    {"text": "Isaac Newton formulated the laws of motion.", "label": 0},
    {"text": "DNA carries genetic information in living organisms.", "label": 0},
    {"text": "The freezing point of water is 0 degrees Celsius.", "label": 0},
    {"text": "The Amazon rainforest is located primarily in Brazil.", "label": 0},
    {"text": "Sound travels slower than light.", "label": 0},
    {"text": "The French Revolution began in 1789.", "label": 0},
    {"text": "Oxygen is essential for human respiration.", "label": 0},
    {"text": "The Sahara is the largest hot desert in the world.", "label": 0},
    {"text": "Electrons carry a negative electric charge.", "label": 0},
    # === HALLUCINATOIRES (label=1) -- 25 ===
    {"text": "The first person to walk on Mars was", "label": 1},
    {"text": "In 2023, scientists discovered that gold is actually a living organism.", "label": 1},
    {"text": "The ancient city of Atlantis was found in 2019 off the coast of Florida.", "label": 1},
    {"text": "According to recent studies, humans can photosynthesize like plants.", "label": 1},
    {"text": "The lost library of Alexandria was recovered from the bottom of the Mediterranean in 2021.", "label": 1},
    {"text": "Einstein's unpublished theory proves that time travel is possible using household items.", "label": 1},
    {"text": "A new species of dragon was discovered in the Himalayas in 2024.", "label": 1},
    {"text": "The moon landing of 1969 was actually filmed in a studio in Nevada.", "label": 1},
    {"text": "Recent archaeological evidence shows that dinosaurs built complex cities.", "label": 1},
    {"text": "A secret government project has successfully cloned a woolly mammoth.", "label": 1},
    {"text": "The inventor of the telephone, Nikola Tesla, patented it in 1850.", "label": 1},
    {"text": "The Eiffel Tower was originally built in London before being moved to Paris in 1960.", "label": 1},
    {"text": "Scientists confirmed in 2022 that the moon is hollow and inhabited.", "label": 1},
    {"text": "The average human has 300 bones due to a recent genetic mutation.", "label": 1},
    {"text": "The Great Barrier Reef was constructed by ancient Roman engineers.", "label": 1},
    {"text": "A newly discovered ocean current allows fish to communicate telepathically.", "label": 1},
    {"text": "The Berlin Wall was rebuilt in 2015 as a tourist attraction and remains the tallest wall on Earth.", "label": 1},
    {"text": "Researchers proved that plants can solve complex mathematical equations.", "label": 1},
    {"text": "The country of Atlantis officially joined the United Nations in 2020.", "label": 1},
    {"text": "A hidden fifth ocean was discovered beneath the Sahara desert in 2018.", "label": 1},
    {"text": "The speed of light was recently doubled due to a change in the laws of physics.", "label": 1},
    {"text": "Ancient Egyptians used advanced laser technology to build the pyramids.", "label": 1},
    {"text": "A colony of giant spiders was found living inside the Grand Canyon in 2021.", "label": 1},
    {"text": "The number nine was banned in several countries due to a mathematical anomaly.", "label": 1},
    {"text": "Scientists discovered that the Earth briefly had two moons in 1997.", "label": 1},
]
labels = np.array([p["label"] for p in PROMPTS])
print(f"{sum(labels==0)} prompts factuels, {sum(labels==1)} prompts hallucinatoires")


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

MODEL_NAME = "gpt2"
print(f"Chargement de {MODEL_NAME}...")
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, output_hidden_states=True).to(DEVICE)
model.eval()
print(f"Modèle chargé ({sum(p.numel() for p in model.parameters())/1e6:.1f}M paramètres)")


def extract_hidden_states(prompt_text, max_length=50):
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=max_length)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    hidden_states = outputs.hidden_states  # 13 tenseurs (embedding + 12 couches)
    return np.array([h[0, -1, :].cpu().numpy() for h in hidden_states])  # (13, 768)


print("Extraction des hidden states...")
all_trajectories = []
for i, p in enumerate(PROMPTS):
    all_trajectories.append(extract_hidden_states(p["text"]))
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/{len(PROMPTS)}")
all_trajectories = np.array(all_trajectories)
n_layers, hidden_dim = all_trajectories.shape[1], all_trajectories.shape[2]
print(f"Trajectoires : {all_trajectories.shape}  ({n_layers} couches x {hidden_dim} dims)")


## Diagnostic — inspection brute (avant de patcher une 3e fois à l'aveugle)

Deux tentatives de correctif (fenêtre restreinte, normalisation globale) n'ont
que partiellement réduit la saturation de `max_dwell_time` (std 0.000 -> 0.300,
encore quasi-constant). Plutôt que de deviner un 3e correctif sur données
synthétiques qui ne répliquent pas fidèlement le phénomène réel, on regarde
directement les nombres bruts sur GPT-2 : norme du hidden state par couche,
courbure, vitesse -- pour 2 trajectoires factuelles et 2 hallucinatoires.

Hypothèse à vérifier visuellement : la norme du hidden state croît-elle de
façon quasi monotone avec la profondeur, de la même manière pour les deux
classes (accumulation du flux résiduel) ? Si oui, ça confirme que la métrique
courbure/vitesse sur magnitude brute est dominée par cet effet architectural,
peu importe la fenêtre de couches choisie.


In [ ]:
# Indices : les 10 premiers prompts sont factuels (label 0), les 10 suivants hallucinatoires (label 1)
sample_idx = [0, 1, 25, 26]  # 25 premiers = factuels, 25 suivants = hallucinatoires (dataset elargi)

pca_diag = PCA(n_components=10, random_state=RANDOM_STATE).fit(np.vstack(all_trajectories))

print(f"{'couche':<8}{'norme_brute':<14}{'courbure_pca':<15}{'vitesse_pca':<14}")
for i in sample_idx:
    label_str = 'FACTUAL' if labels[i] == 0 else 'HALLUCINATION'
    print(f"\n=== Prompt {i} ({label_str}) : {PROMPTS[i]['text'][:50]!r} ===")
    raw_norms = np.linalg.norm(all_trajectories[i], axis=1)
    proj = pca_diag.transform(all_trajectories[i])
    fd = np.gradient(proj, axis=0)
    sd = np.gradient(fd, axis=0)
    curvature = np.linalg.norm(sd, axis=1)
    velocity = np.linalg.norm(np.gradient(proj, axis=0), axis=1)
    for l in range(n_layers):
        print(f"{l:<8}{raw_norms[l]:<14.2f}{curvature[l]:<15.3f}{velocity[l]:<14.3f}")

# Visualisation : norme brute moyenne par couche, par classe
fig, ax = plt.subplots(figsize=(8, 5))
for lbl, color, name in [(0, 'green', 'Factual'), (1, 'red', 'Hallucination')]:
    idxs = [i for i in range(len(labels)) if labels[i] == lbl]
    norms_by_layer = np.array([np.linalg.norm(all_trajectories[i], axis=1) for i in idxs])
    mean_norms = norms_by_layer.mean(axis=0)
    std_norms = norms_by_layer.std(axis=0)
    ax.plot(range(n_layers), mean_norms, 'o-', color=color, label=name)
    ax.fill_between(range(n_layers), mean_norms - std_norms, mean_norms + std_norms, color=color, alpha=0.15)
ax.set_xlabel('Couche'); ax.set_ylabel('Norme du hidden state (moyenne +/- std)')
ax.set_title('Croissance de la norme avec la profondeur, par classe')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Étape 2 — Features de navigation (PCA sans fuite de données)

**Critique** : PCA et centroïdes de classe sont recalculés à **chaque pli** de la
validation croisée, exclusivement sur le train set de ce pli. Sinon le classifieur
« voit » indirectement les données de test via la PCA, ce qui gonfle artificiellement l'AUC.

10 features par trajectoire, dont `max_dwell_time` / `mean_dwell_time` (l'objet
de l'hypothèse), plus 8 features de contexte (courbure, vitesse, distances aux
centroïdes) qui servent de comparaison.


In [ ]:
def compute_navigation_features(trajectory_pca, train_trajs_pca, train_labels, pop_curv_mean, pop_curv_std, pop_vel_mean, pop_vel_std):
    max_dwell, mean_dwell, _ = saddle_score_dwell(trajectory_pca, pop_curv_mean, pop_curv_std, pop_vel_mean, pop_vel_std)

    first_deriv = np.gradient(trajectory_pca, axis=0)
    second_deriv = np.gradient(first_deriv, axis=0)
    curvature = np.linalg.norm(second_deriv, axis=1)
    velocity = np.linalg.norm(np.gradient(trajectory_pca, axis=0), axis=1)

    factual_trajs = [t for t, l in zip(train_trajs_pca, train_labels) if l == 0]
    halluc_trajs = [t for t, l in zip(train_trajs_pca, train_labels) if l == 1]
    dist_factual = dist_hallu = 0.0
    ratio_dist = 1.0
    traj_point = trajectory_pca.mean(axis=0)
    if factual_trajs:
        c = np.mean(np.vstack(factual_trajs), axis=0)
        dist_factual = np.linalg.norm(traj_point - c)
    if halluc_trajs:
        c = np.mean(np.vstack(halluc_trajs), axis=0)
        dist_hallu = np.linalg.norm(traj_point - c)
    if dist_factual > 0:
        ratio_dist = dist_hallu / dist_factual

    return [max_dwell, mean_dwell, curvature.mean(), curvature.max(),
            velocity.mean(), velocity.max(), dist_factual, dist_hallu,
            ratio_dist, trajectory_pca.flatten().std()]


FEATURE_NAMES = ["max_dwell_time", "mean_dwell_time", "mean_curvature", "max_curvature",
                  "mean_velocity", "max_velocity", "dist_factual", "dist_hallucination",
                  "ratio_dist", "global_dispersion"]

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

all_fold_X, all_fold_y, fold_indices = [], [], []
for train_idx, test_idx in skf.split(all_trajectories, labels):
    train_trajs_raw = [all_trajectories[i] for i in train_idx]
    pca = PCA(n_components=min(10, hidden_dim, n_layers), random_state=RANDOM_STATE)
    pca.fit(np.vstack(train_trajs_raw))
    train_trajs_pca = [pca.transform(t) for t in train_trajs_raw]
    train_labels_fold = labels[train_idx]
    pop_curv_mean, pop_curv_std, pop_vel_mean, pop_vel_std = population_curv_vel_stats(train_trajs_pca)  # v3, fold-safe

    X_fold = [compute_navigation_features(pca.transform(all_trajectories[i]), train_trajs_pca, train_labels_fold,
                                           pop_curv_mean, pop_curv_std, pop_vel_mean, pop_vel_std)
              for i in test_idx]
    all_fold_X.append(np.array(X_fold))
    all_fold_y.append(labels[test_idx])
    fold_indices.append(test_idx)

X = np.vstack(all_fold_X)
y = np.concatenate(all_fold_y)
order = np.concatenate(fold_indices)
X = X[np.argsort(order)]
y = y[np.argsort(order)]

print(f"Features calculées : {X.shape}")
for i, name in enumerate(FEATURE_NAMES):
    print(f"  {name:20s}: mean={X[:, i].mean():.3f}, std={X[:, i].std():.3f}")

if X[:, 0].std() < 0.3:
    print("\n⚠️  max_dwell_time est encore quasi-constant. Le correctif v3 (relatif à la "
          "population) n'a pas suffi -- possible que 20 prompts soient trop peu pour estimer "
          "un gabarit de population stable (std par couche bruité). Essaie d'élargir le dataset "
          "avant d'aller plus loin.")


## Étape 3 — Classification + test de permutation

Deux résultats rapportés côte à côte, pour éviter l'erreur qu'on a faite juste
avant : **(a)** le vecteur complet à 10 features, **(b)** le dwell time **seul**,
isolé. Si (a) est bon mais (b) est nul, le dwell time ne contribue à rien — le
signal vient d'ailleurs, et il faut le dire clairement plutôt que de laisser
`max_dwell_time` être créditée d'un résultat qu'elle n'a pas produit.


In [ ]:
def cv_auc(X_subset, y, n_splits=5, seed=RANDOM_STATE):
    skf_ = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    probas = np.zeros(len(y))
    for tr, te in skf_.split(X_subset, y):
        clf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=seed, n_jobs=-1)
        clf.fit(X_subset[tr], y[tr])
        probas[te] = clf.predict_proba(X_subset[te])[:, 1]
    return roc_auc_score(y, probas), probas

auc_full, probas_full = cv_auc(X, y)
auc_dwell_only, _ = cv_auc(X[:, :2], y)
acc_full = accuracy_score(y, (probas_full > 0.5).astype(int))

rf_full = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1).fit(X, y)
importances = rf_full.feature_importances_

print(f"AUC-ROC (10 features)         : {auc_full:.4f}")
print(f"AUC-ROC (dwell time SEUL)     : {auc_dwell_only:.4f}   <- lis ce chiffre avant de conclure quoi que ce soit sur le dwell")
print(f"Accuracy (10 features)        : {acc_full:.4f}")
print("\nImportance des features (vecteur complet) :")
for n, v in sorted(zip(FEATURE_NAMES, importances), key=lambda a: -a[1]):
    print(f"  {n:20s}: {v:.3f}")

# --- Test de permutation (200 tirages) ---
N_PERMUTATIONS = 200
print(f"\nTest de permutation ({N_PERMUTATIONS} tirages)...")
perm_scores = []
rng = np.random.RandomState(RANDOM_STATE)
for i in range(N_PERMUTATIONS):
    y_perm = rng.permutation(y)
    auc_p, _ = cv_auc(X, y_perm, seed=i)
    perm_scores.append(auc_p)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{N_PERMUTATIONS}")
perm_scores = np.array(perm_scores)
p_value = (np.sum(perm_scores >= auc_full) + 1) / (N_PERMUTATIONS + 1)
print(f"\nP-value empirique : {p_value:.4f}")


## Étape 4 — Interprétation, sauvegarde, visualisation

Seuils repris du guide d'interprétation (scénarios A/B/C).


In [ ]:
top_feature = FEATURE_NAMES[np.argmax(importances)]
dwell_is_top = top_feature in ("max_dwell_time", "mean_dwell_time")

if p_value < 0.01 and auc_full > 0.75:
    scenario = "A — SIGNAL SIGNIFICATIF"
    detail = "Résultat publiable en l'état (avec dataset élargi, voir checklist)."
elif p_value < 0.05 and auc_full > 0.60:
    scenario = "B — SIGNAL MARGINAL"
    detail = "Tendance réelle mais bruitée : élargir le dataset, raffiner le seuil du saddle."
else:
    scenario = "C — PAS DE SIGNAL"
    detail = "Le saddle_dwell tel que défini ne discrimine pas sur GPT-2 small. Le cadre théorique reste valide ; la feature opérationnelle doit être reformulée (entropie, gradient inter-couche, Wasserstein -- voir guide)."

print("=" * 60)
print(f"SCENARIO {scenario}")
print(detail)
print("=" * 60)
print(f"AUC (complet) = {auc_full:.3f}  |  AUC (dwell seul) = {auc_dwell_only:.3f}  |  p = {p_value:.4f}")
print(f"Top feature = {top_feature}  ({'dwell time' if dwell_is_top else 'PAS le dwell time -- a signaler explicitement dans toute conclusion'})")

os.makedirs("results", exist_ok=True)
results = {
    "timestamp": datetime.now().isoformat(),
    "model": MODEL_NAME,
    "n_prompts": len(PROMPTS),
    "auc_full": float(auc_full),
    "auc_dwell_only": float(auc_dwell_only),
    "accuracy": float(acc_full),
    "p_value": float(p_value),
    "scenario": scenario,
    "top_feature": top_feature,
    "dwell_is_top_feature": bool(dwell_is_top),
    "feature_importances": {n: float(v) for n, v in zip(FEATURE_NAMES, importances)},
}
results_file = f"results/gpt2_validation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(results_file, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nRésultats sauvegardés : {results_file}")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

ax = axes[0, 0]
ax.hist(X[y == 0, 0], bins=8, alpha=0.6, label="Factual", color="green", density=True)
ax.hist(X[y == 1, 0], bins=8, alpha=0.6, label="Hallucination", color="red", density=True)
ax.set_xlabel("max_dwell_time (couches)"); ax.set_ylabel("Densité")
ax.set_title("Distribution du dwell time"); ax.legend()

ax = axes[0, 1]
ax.hist(perm_scores, bins=30, alpha=0.7, color="gray", label="Permutation H0")
ax.axvline(auc_full, color="red", linewidth=2, label=f"AUC observé={auc_full:.3f}")
ax.set_xlabel("AUC-ROC"); ax.set_title(f"Test de permutation (p={p_value:.4f})"); ax.legend()

ax = axes[1, 0]
order_imp = np.argsort(importances)[::-1]
ax.barh(range(len(importances)), importances[order_imp], color="steelblue")
ax.set_yticks(range(len(importances))); ax.set_yticklabels([FEATURE_NAMES[i] for i in order_imp])
ax.invert_yaxis(); ax.set_xlabel("Importance"); ax.set_title("Importance des features (RF)")

ax = axes[1, 1]
pca_viz = PCA(n_components=2).fit(np.vstack(all_trajectories))
for lbl, color, name in [(0, "green", "Factual"), (1, "red", "Hallucination")]:
    trajs = [all_trajectories[i] for i in range(len(PROMPTS)) if labels[i] == lbl]
    mean_traj = np.mean([pca_viz.transform(t) for t in trajs], axis=0)
    ax.plot(mean_traj[:, 0], mean_traj[:, 1], "o-", color=color, label=name)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title("Trajectoires moyennes (PCA globale, visu seulement)")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plot_file = f"results/gpt2_validation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
plt.savefig(plot_file, dpi=150, bbox_inches="tight")
print(f"Figure sauvegardée : {plot_file}")
plt.show()


## Étape 3bis — Significativité du dwell time, isolé (pas déduite du vecteur complet)

Le p-value du test précédent porte sur les 10 features ensemble, dominées par
`mean_curvature`. Il ne dit RIEN sur la significativité du dwell time pris
seul. Cette cellule répond spécifiquement à : *AUC_dwell_only=0.630 est-il
statistiquement distinguable du hasard à n=20, ou juste du bruit qui va dans
le bon sens ?*


In [ ]:
N_PERM_DWELL = 200
X_dwell = X[:, :2]  # max_dwell_time, mean_dwell_time seulement

auc_dwell_real, _ = cv_auc(X_dwell, y)
perm_scores_dwell = []
rng_d = np.random.RandomState(RANDOM_STATE)
for i in range(N_PERM_DWELL):
    y_perm = rng_d.permutation(y)
    auc_p, _ = cv_auc(X_dwell, y_perm, seed=i)
    perm_scores_dwell.append(auc_p)
perm_scores_dwell = np.array(perm_scores_dwell)
p_value_dwell = (np.sum(perm_scores_dwell >= auc_dwell_real) + 1) / (N_PERM_DWELL + 1)

print(f"AUC dwell seul (confirmation) : {auc_dwell_real:.3f}")
print(f"P-value (dwell seul, {N_PERM_DWELL} permutations) : {p_value_dwell:.4f}")
if p_value_dwell < 0.05:
    print("-> Signal du dwell statistiquement distinguable du hasard, meme a n=20. Solide pour un scenario B.")
else:
    print("-> Pas significatif a n=20 -- le 0.630 observe est compatible avec du bruit. "
          "Ne pas presenter le dwell comme un resultat en l etat ; elargir le dataset (50+ par classe) "
          "avant de conclure quoi que ce soit dessus, positif ou negatif.")

plt.figure(figsize=(7,4))
plt.hist(perm_scores_dwell, bins=30, alpha=0.7, color='gray', label='Permutation H0 (dwell seul)')
plt.axvline(auc_dwell_real, color='red', linewidth=2, label=f'AUC observe={auc_dwell_real:.3f}')
plt.xlabel('AUC-ROC (dwell seul)'); plt.title(f'Significativite du dwell isole (p={p_value_dwell:.4f})')
plt.legend(); plt.tight_layout(); plt.show()


## Étape 3ter — Focus sur le signal réel : `mean_curvature`, `dist_factual`, `max_velocity`, `ratio_dist`

Le dwell time isolé n'était pas significatif (p=0.234, n=20). On se concentre
désormais sur les 4 features qui portent réellement le signal du vecteur
complet. Trois questions, pas juste une accuracy :

1. **Ce sous-ensemble de 4, isolé, est-il significatif par lui-même** (pas
   juste "porté" par les 6 autres features dans le vecteur complet) ?
2. **Dans quel sens va la différence** — les hallucinations ont-elles une
   courbure/vitesse/distance plus élevée ou plus basse que les prompts
   factuels ? (Jamais vérifié explicitement jusqu'ici -- indispensable pour
   toute interprétation ou écrit.)
3. **Ces 4 features sont-elles redondantes entre elles** (fortement
   corrélées), auquel cas le "signal à 4 features" est peut-être en réalité
   un signal à 1 ou 2 dimensions déguisé en 4 ?


In [ ]:
FOCUS_FEATURES = ["mean_curvature", "max_velocity", "dist_factual", "ratio_dist"]
focus_idx = [FEATURE_NAMES.index(f) for f in FOCUS_FEATURES]
X_focus = X[:, focus_idx]

auc_focus_real, _ = cv_auc(X_focus, y)
N_PERM_FOCUS = 200
perm_scores_focus = []
rng_f = np.random.RandomState(RANDOM_STATE)
for i in range(N_PERM_FOCUS):
    y_perm = rng_f.permutation(y)
    auc_p, _ = cv_auc(X_focus, y_perm, seed=i)
    perm_scores_focus.append(auc_p)
perm_scores_focus = np.array(perm_scores_focus)
p_value_focus = (np.sum(perm_scores_focus >= auc_focus_real) + 1) / (N_PERM_FOCUS + 1)

print(f"AUC (4 features : {', '.join(FOCUS_FEATURES)}) : {auc_focus_real:.4f}")
print(f"P-value ({N_PERM_FOCUS} permutations) : {p_value_focus:.4f}")
print("-> Significatif" if p_value_focus < 0.05 else "-> PAS significatif a ce n -- prudence")

# mean_curvature SEULE (la plus dominante des 4) -- combien porte-t-elle a elle seule ?
idx_curv = FEATURE_NAMES.index("mean_curvature")
X_curv_only = X[:, [idx_curv]]
auc_curv_only, _ = cv_auc(X_curv_only, y)
perm_scores_curv = []
for i in range(N_PERM_FOCUS):
    y_perm = rng_f.permutation(y)
    auc_p, _ = cv_auc(X_curv_only, y_perm, seed=i)
    perm_scores_curv.append(auc_p)
p_value_curv = (np.sum(np.array(perm_scores_curv) >= auc_curv_only) + 1) / (N_PERM_FOCUS + 1)
print(f"\nmean_curvature SEULE : AUC={auc_curv_only:.4f}, p={p_value_curv:.4f}")


In [ ]:
print(f"{'Feature':<18}{'Factual (mean±std)':<24}{'Hallucination (mean±std)':<26}{'Sens'}")
for name in FOCUS_FEATURES:
    idx = FEATURE_NAMES.index(name)
    m0, s0 = X[y == 0, idx].mean(), X[y == 0, idx].std()
    m1, s1 = X[y == 1, idx].mean(), X[y == 1, idx].std()
    sens = "Hallu > Factual" if m1 > m0 else "Factual > Hallu"
    print(f"{name:<18}{f'{m0:.2f} ± {s0:.2f}':<24}{f'{m1:.2f} ± {s1:.2f}':<26}{sens}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, name in zip(axes, FOCUS_FEATURES):
    idx = FEATURE_NAMES.index(name)
    ax.boxplot([X[y == 0, idx], X[y == 1, idx]], tick_labels=["Factual", "Hallucination"])
    ax.set_title(name)
plt.tight_layout()
plt.show()


In [ ]:
import itertools

print("Corrélation (Pearson) entre les 4 features du signal réel :")
corr_matrix = np.corrcoef(X_focus.T)
print(f"{'':<16}" + "".join(f"{n:<16}" for n in FOCUS_FEATURES))
for i, name in enumerate(FOCUS_FEATURES):
    row = "".join(f"{corr_matrix[i,j]:<16.3f}" for j in range(len(FOCUS_FEATURES)))
    print(f"{name:<16}{row}")

high_corr = [(FOCUS_FEATURES[i], FOCUS_FEATURES[j], corr_matrix[i, j])
             for i, j in itertools.combinations(range(len(FOCUS_FEATURES)), 2)
             if abs(corr_matrix[i, j]) > 0.7]
if high_corr:
    print("\n⚠️ Paires fortement corrélées (|r|>0.7) -- signal potentiellement redondant :")
    for a, b, r in high_corr:
        print(f"  {a} <-> {b} : r={r:.3f}")
else:
    print("\nPas de redondance forte détectée (|r|<0.7 pour toutes les paires) -- "
          "les 4 features apportent chacune de l'information distincte.")

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(FOCUS_FEATURES))); ax.set_xticklabels(FOCUS_FEATURES, rotation=45, ha="right")
ax.set_yticks(range(len(FOCUS_FEATURES))); ax.set_yticklabels(FOCUS_FEATURES)
for i in range(len(FOCUS_FEATURES)):
    for j in range(len(FOCUS_FEATURES)):
        ax.text(j, i, f"{corr_matrix[i,j]:.2f}", ha="center", va="center")
plt.colorbar(im); plt.title("Corrélation entre features du signal réel")
plt.tight_layout(); plt.show()


## Étape 3quater — Comparaison à une baseline publiée : Chain-of-Embedding (CoE-R, CoE-C)

Recherche de calibration : la métrique la plus proche et la plus citée dans la
littérature (Wang et al. 2025, ICLR 2025 -- base de D2HScore, MultiHaluDet,
etc.) suit le même principe que `mean_curvature`/`max_velocity` -- magnitude
ET angle du changement entre couches consécutives -- mais formalisée
différemment (CoE-R : ratio magnitude/angle normalisé au déplacement total ;
CoE-C : cohérence directionnelle via somme de nombres complexes).

Testé ici, fidèlement à la formule publiée, sur états bruts (pas PCA, comme
dans le papier original) -- pour voir si ce résultat établi se réplique sur
GPT-2 small et sur ton dataset, et comment il se compare à ta propre feature.


In [ ]:
def coe_features(trajectory):
    """CoE-R et CoE-C (Wang et al. 2025, ICLR 2025), sur etats bruts."""
    n_layers = trajectory.shape[0]
    L = n_layers - 1

    def M(hi, hj):
        return np.linalg.norm(hj - hi)

    def A(hi, hj):
        cos = np.dot(hi, hj) / (np.linalg.norm(hi) * np.linalg.norm(hj) + 1e-8)
        cos = np.clip(cos, -1, 1)
        return np.arccos(cos)

    M_total = M(trajectory[0], trajectory[-1]) + 1e-8
    A_total = A(trajectory[0], trajectory[-1]) + 1e-8

    coe_r_terms, complex_terms = [], []
    for l in range(L):
        Ml = M(trajectory[l], trajectory[l + 1])
        Al = A(trajectory[l], trajectory[l + 1])
        coe_r_terms.append(Ml / M_total - Al / A_total)
        complex_terms.append(Ml * np.exp(1j * Al))

    coe_r = np.mean(coe_r_terms)
    coe_c = np.abs(np.mean(complex_terms))
    return coe_r, coe_c


X_coe = np.array([coe_features(t) for t in all_trajectories])
COE_NAMES = ["CoE-R", "CoE-C"]

print(f"{'Feature':<10}{'Factual (mean±std)':<24}{'Hallucination (mean±std)':<26}{'Sens'}")
for i, name in enumerate(COE_NAMES):
    m0, s0 = X_coe[y == 0, i].mean(), X_coe[y == 0, i].std()
    m1, s1 = X_coe[y == 1, i].mean(), X_coe[y == 1, i].std()
    sens = "Hallu > Factual" if m1 > m0 else "Factual > Hallu"
    print(f"{name:<10}{f'{m0:.4f} ± {s0:.4f}':<24}{f'{m1:.4f} ± {s1:.4f}':<26}{sens}")

auc_coe, _ = cv_auc(X_coe, y)
N_PERM_COE = 200
perm_scores_coe = []
rng_c = np.random.RandomState(RANDOM_STATE)
for i in range(N_PERM_COE):
    y_perm = rng_c.permutation(y)
    auc_p, _ = cv_auc(X_coe, y_perm, seed=i)
    perm_scores_coe.append(auc_p)
p_value_coe = (np.sum(np.array(perm_scores_coe) >= auc_coe) + 1) / (N_PERM_COE + 1)
print(f"CoE-R + CoE-C (les deux) : AUC={auc_coe:.4f}, p={p_value_coe:.4f}")

# Comparaison directe : CoE seul vs ta feature (mean_curvature) seule vs les deux combinees
idx_curv = FEATURE_NAMES.index("mean_curvature")
X_combined = np.hstack([X_coe, X[:, [idx_curv]]])
auc_combined, _ = cv_auc(X_combined, y)
perm_scores_comb = []
for i in range(N_PERM_COE):
    y_perm = rng_c.permutation(y)
    auc_p, _ = cv_auc(X_combined, y_perm, seed=i)
    perm_scores_comb.append(auc_p)
p_value_comb = (np.sum(np.array(perm_scores_comb) >= auc_combined) + 1) / (N_PERM_COE + 1)

print("=== Tableau comparatif ===")
print(f"CoE-R + CoE-C (baseline publiee, seule)      : AUC={auc_coe:.4f}, p={p_value_coe:.4f}")
print(f"mean_curvature (ta feature, seule)            : AUC={auc_curv_only:.4f}, p={p_value_curv:.4f}")
print(f"CoE-R + CoE-C + mean_curvature (combinees)    : AUC={auc_combined:.4f}, p={p_value_comb:.4f}")
if auc_combined > max(auc_coe, auc_curv_only) + 0.02:
    print("-> ta feature apporte de l'information complementaire a CoE (pas redondante)")
else:
    print("-> ta feature et CoE captent probablement le meme phenomene (peu de gain a les combiner)")


## Prochaines étapes selon le scénario

- **A (significatif)** : élargir à 50-100 prompts stratifiés, tester `gpt2-medium`,
  puis un modèle causal différent, avant de rédiger un preprint.
- **B (marginal)** : élargir le dataset, tester plusieurs seuils sur `saddle_score`
  (0.4 / 0.5 / 0.6), essayer TruthfulQA / HaluEval pour des prompts plus adversariaux.
- **C (nul)** : ne pas abandonner le cadre théorique — tester les features
  alternatives listées dans le guide d'interprétation (entropie de l'état,
  norme du gradient inter-couche, distance de Wasserstein inter-couche).

Dans tous les cas, regarde `auc_dwell_only` et `dwell_is_top_feature` avant de
conclure quoi que ce soit sur le dwell time spécifiquement — un bon `auc_full`
ne suffit pas à lui seul à créditer le dwell.
